# Venezuela Land Cover Change Analysis (2017–2024)

Sentinel-2 10m Land Use/Land Cover — ESRI Living Atlas  
Projection: Albers Equal-Area Conic (custom, centred on Venezuela)

This notebook loads the CSV outputs from `analyze_cover.py` and produces
static (matplotlib/seaborn) and interactive (plotly) figures for reporting.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from pathlib import Path

In [ ]:
# --- Paths ---
RESULTS = Path("..") / "outputs" / "results"
FIGURES = Path("..") / "outputs" / "figures"
FIGURES.mkdir(parents=True, exist_ok=True)

In [ ]:
# --- Load data ---
cover_2017 = pd.read_csv(RESULTS / "cover_2017.csv")
cover_2024 = pd.read_csv(RESULTS / "cover_2024.csv")
comparison = pd.read_csv(RESULTS / "comparison_2017_2024.csv")
matrix_raw = pd.read_csv(RESULTS / "transition_matrix_km2.csv", index_col=0)

In [ ]:
# --- Filter out nodata and cloud classes ---
EXCLUDE_IDS = {0, 3, 6, 10}

c17 = cover_2017[~cover_2017["class_id"].isin(EXCLUDE_IDS)].copy()
c24 = cover_2024[~cover_2024["class_id"].isin(EXCLUDE_IDS)].copy()
comp = comparison[~comparison["class_id"].isin(EXCLUDE_IDS)].copy()

# Drop nodata/cloud rows and columns from transition matrix
exclude_labels = ["No data", "Clouds"]
matrix = matrix_raw.drop(
    index=[l for l in exclude_labels if l in matrix_raw.index],
    columns=[l for l in exclude_labels if l in matrix_raw.columns],
    errors="ignore",
)

In [ ]:
# --- Consistent color palette across all charts ---
# Follows ESRI LULC conventions where possible
CLASS_COLORS = {
    "Water":              "#1A5BAB",
    "Trees":              "#397D49",
    "Flooded vegetation": "#7A87C6",
    "Crops":              "#E49635",
    "Built area":         "#E02020",
    "Bare ground":        "#B39F81",
    "Snow/Ice":           "#B4D5F0",
    "Rangeland":          "#DFC35A",
}

# Matplotlib style
plt.rcParams.update({
    "figure.facecolor": "#FAFAFA",
    "axes.facecolor": "#FAFAFA",
    "axes.edgecolor": "#333333",
    "axes.grid": True,
    "grid.alpha": 0.3,
    "font.size": 11,
})

---
## 1. Land cover composition by year

In [ ]:
# Quick overview tables
print("2017 — valid classes")
display(c17[["class_name", "area_km2", "area_ha", "pct"]])
print("\n2024 — valid classes")
display(c24[["class_name", "area_km2", "area_ha", "pct"]])

### 1a. Grouped bar chart — area per class (static)

In [ ]:
fig, ax = plt.subplots(figsize=(12, 6))

classes = comp["class_name"].values
x = np.arange(len(classes))
w = 0.35

bars_17 = ax.bar(x - w/2, comp["area_km2_2017"], w, label="2017", color="#4A7C59", edgecolor="white")
bars_24 = ax.bar(x + w/2, comp["area_km2_2024"], w, label="2024", color="#D4863B", edgecolor="white")

ax.set_xticks(x)
ax.set_xticklabels(classes, rotation=35, ha="right")
ax.set_ylabel("Area (km²)")
ax.set_title("Land Cover by Class — 2017 vs 2024")
ax.legend()
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda v, _: f"{v:,.0f}"))

fig.tight_layout()
fig.savefig(FIGURES / "bar_cover_comparison.png", dpi=200)
plt.show()

### 1b. Grouped bar chart (interactive)

In [ ]:
df_plot = pd.concat([
    c17[["class_name", "area_km2"]].assign(year="2017"),
    c24[["class_name", "area_km2"]].assign(year="2024"),
])

fig = px.bar(
    df_plot, x="class_name", y="area_km2", color="year",
    barmode="group",
    color_discrete_map={"2017": "#4A7C59", "2024": "#D4863B"},
    labels={"area_km2": "Area (km²)", "class_name": "", "year": "Year"},
    title="Land Cover by Class — 2017 vs 2024",
)
fig.update_layout(template="plotly_white")
fig.show()

---
## 2. Donut charts — class proportions

In [ ]:
# Static donut — side by side
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

for ax, df, year in [(axes[0], c17, "2017"), (axes[1], c24, "2024")]:
    colors = [CLASS_COLORS.get(name, "#999999") for name in df["class_name"]]
    wedges, texts, autotexts = ax.pie(
        df["area_km2"], labels=df["class_name"], colors=colors,
        autopct=lambda p: f"{p:.1f}%" if p > 3 else "",
        pctdistance=0.8, startangle=90,
        wedgeprops={"width": 0.45, "edgecolor": "white", "linewidth": 1.5},
    )
    for t in autotexts:
        t.set_fontsize(9)
    ax.set_title(year, fontsize=14, fontweight="bold")

fig.suptitle("Land Cover Composition", fontsize=15, y=1.02)
fig.tight_layout()
fig.savefig(FIGURES / "donut_cover.png", dpi=200, bbox_inches="tight")
plt.show()

In [ ]:
# Interactive donut — 2017
fig17 = px.pie(
    c17, values="area_km2", names="class_name", hole=0.45,
    color="class_name", color_discrete_map=CLASS_COLORS,
    title="Land Cover 2017",
)
fig17.update_traces(textinfo="percent+label", textposition="inside")
fig17.update_layout(template="plotly_white", showlegend=False)
fig17.show()

# Interactive donut — 2024
fig24 = px.pie(
    c24, values="area_km2", names="class_name", hole=0.45,
    color="class_name", color_discrete_map=CLASS_COLORS,
    title="Land Cover 2024",
)
fig24.update_traces(textinfo="percent+label", textposition="inside")
fig24.update_layout(template="plotly_white", showlegend=False)
fig24.show()

---
## 3. Net change per class

In [ ]:
# Sort by magnitude for visual clarity
comp_sorted = comp.sort_values("change_km2")

colors = ["#B83230" if v < 0 else "#2D7D46" for v in comp_sorted["change_km2"]]

fig, ax = plt.subplots(figsize=(10, 6))
ax.barh(comp_sorted["class_name"], comp_sorted["change_km2"], color=colors, edgecolor="white")
ax.axvline(0, color="#333333", linewidth=0.8)
ax.set_xlabel("Net Change (km²)")
ax.set_title("Net Land Cover Change — 2017 to 2024")
ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda v, _: f"{v:,.0f}"))

# Value labels
for i, (val, name) in enumerate(zip(comp_sorted["change_km2"], comp_sorted["class_name"])):
    ha = "left" if val >= 0 else "right"
    offset = 200 if val >= 0 else -200
    ax.text(val + offset, i, f"{val:+,.0f}", va="center", ha=ha, fontsize=9)

fig.tight_layout()
fig.savefig(FIGURES / "bar_net_change.png", dpi=200)
plt.show()

In [ ]:
# Interactive version
fig = px.bar(
    comp_sorted, x="change_km2", y="class_name", orientation="h",
    color=comp_sorted["change_km2"].apply(lambda v: "Loss" if v < 0 else "Gain"),
    color_discrete_map={"Loss": "#B83230", "Gain": "#2D7D46"},
    labels={"change_km2": "Net Change (km²)", "class_name": "", "color": ""},
    title="Net Land Cover Change — 2017 to 2024",
)
fig.update_layout(template="plotly_white", showlegend=True)
fig.show()

---
## 4. Transition matrix heatmap

In [ ]:
# Off-diagonal only (exclude persistence on the diagonal for better contrast)
matrix_offdiag = matrix.copy()
np.fill_diagonal(matrix_offdiag.values, np.nan)

fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(
    matrix_offdiag, annot=True, fmt=".0f", cmap="YlOrRd",
    linewidths=0.5, linecolor="white",
    cbar_kws={"label": "Area (km²)", "shrink": 0.8},
    ax=ax,
)
ax.set_title("Transition Matrix — Off-diagonal (2017 → 2024)")
ax.set_ylabel("2017")
ax.set_xlabel("2024")

fig.tight_layout()
fig.savefig(FIGURES / "heatmap_transitions.png", dpi=200)
plt.show()

In [ ]:
# Full matrix including persistence (interactive)
fig = px.imshow(
    matrix, text_auto=".0f", color_continuous_scale="YlOrRd",
    labels={"x": "2024", "y": "2017", "color": "km²"},
    title="Full Transition Matrix (2017 → 2024)",
    aspect="auto",
)
fig.update_layout(template="plotly_white", width=700, height=600)
fig.show()

---
## 5. Sankey diagram — major class-to-class flows

Showing only transitions above a minimum threshold to keep the diagram readable.

In [ ]:
# Build flow data from the transition matrix
# Only off-diagonal flows (actual transitions, not persistence)
flows = []
for src_class in matrix.index:
    for dst_class in matrix.columns:
        if src_class == dst_class:
            continue
        val = matrix.loc[src_class, dst_class]
        if val > 0:
            flows.append({"source": src_class, "target": dst_class, "value": val})

flows_df = pd.DataFrame(flows)

# Keep only the top flows (adjustable threshold)
threshold = flows_df["value"].quantile(0.75)
flows_df = flows_df[flows_df["value"] >= threshold].copy()

print(f"Showing {len(flows_df)} flows above {threshold:,.0f} km²")
flows_df.sort_values("value", ascending=False).head(10)

In [ ]:
# Build node list: source nodes (2017) on left, target nodes (2024) on right
source_labels = [f"{s} (2017)" for s in flows_df["source"].unique()]
target_labels = [f"{t} (2024)" for t in flows_df["target"].unique()]
all_labels = source_labels + target_labels

# Map names to indices
src_map = {name: i for i, name in enumerate(source_labels)}
tgt_map = {name: i for i, name in enumerate(target_labels, start=len(source_labels))}

# Node colors
node_colors = (
    [CLASS_COLORS.get(s, "#AAAAAA") for s in flows_df["source"].unique()] +
    [CLASS_COLORS.get(t, "#AAAAAA") for t in flows_df["target"].unique()]
)

# Link colors (semi-transparent source color)
def hex_to_rgba(hex_color, alpha=0.35):
    h = hex_color.lstrip("#")
    r, g, b = int(h[0:2], 16), int(h[2:4], 16), int(h[4:6], 16)
    return f"rgba({r},{g},{b},{alpha})"

link_sources = [src_map[f"{row.source} (2017)"] for row in flows_df.itertuples()]
link_targets = [tgt_map[f"{row.target} (2024)"] for row in flows_df.itertuples()]
link_values = flows_df["value"].tolist()
link_colors = [
    hex_to_rgba(CLASS_COLORS.get(row.source, "#AAAAAA"))
    for row in flows_df.itertuples()
]

fig = go.Figure(go.Sankey(
    node=dict(
        pad=20, thickness=25,
        label=all_labels,
        color=node_colors,
    ),
    link=dict(
        source=link_sources,
        target=link_targets,
        value=link_values,
        color=link_colors,
    ),
))

fig.update_layout(
    title="Land Cover Transitions — 2017 to 2024 (major flows)",
    template="plotly_white",
    font_size=12,
    width=900, height=550,
)
fig.show()

---
## 6. Summary table

Formatted comparison table ready for export or copy-paste into a report.

In [ ]:
summary = comp[[
    "class_name", "area_km2_2017", "area_km2_2024",
    "change_km2", "change_pct"
]].copy()

summary.columns = ["Class", "2017 (km²)", "2024 (km²)", "Change (km²)", "Change (%)"]
summary = summary.sort_values("Change (km²)")

# Style for notebook display
display(
    summary.style
    .format({"2017 (km²)": "{:,.0f}", "2024 (km²)": "{:,.0f}",
             "Change (km²)": "{:+,.0f}", "Change (%)": "{:+.1f}%"})
    .bar(subset=["Change (km²)"], align="zero",
         color=["#B83230", "#2D7D46"])
    .set_caption("Land Cover Change Summary — Venezuela 2017 to 2024")
)

---

**Next steps:**
- Load the rasters in QGIS for spatial visualization and cartographic layout
- Cross the change raster with protected areas, watersheds, or administrative boundaries
- Export figures from `outputs/figures/` for the final report